In [1]:
# ================================================================
# STEP 5: HYBRID AI CAREER RECOMMENDATION ENGINE
# ================================================================

import pandas as pd
import numpy as np
import os
import re

# ================================================================
# 1. PROJECT PATHS
# ================================================================

BASE_PATH = r"C:\Users\shiva\Desktop\AI_Career_Recommendation_System"

CAREER_PATH = os.path.join(
    BASE_PATH,
    "data",
    "career_knowledge_base.csv"
)

JOB_PATH = os.path.join(
    BASE_PATH,
    "data",
    "processed_jobs.csv"
)

# ================================================================
# 2. LOAD DATA
# ================================================================

careers = pd.read_csv(CAREER_PATH)
jobs = pd.read_csv(JOB_PATH)

print("=" * 70)
print("HYBRID AI CAREER RECOMMENDATION ENGINE")
print("=" * 70)

print("\nCareer Knowledge Base :", careers.shape)
print("Job Dataset           :", jobs.shape)

# ================================================================
# 3. HELPER FUNCTIONS
# ================================================================

def clean_text(text):
    """
    Convert text into lowercase and remove extra spaces.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def convert_to_set(text):
    """
    Convert comma-separated values into a Python set.
    """
    if pd.isna(text):
        return set()

    return {
        clean_text(item)
        for item in str(text).split(",")
        if clean_text(item)
    }


def calculate_match(user_items, career_items):
    """
    Calculate percentage of user's selected items
    that match the career requirements.
    """

    user_items = set(user_items)
    career_items = set(career_items)

    if len(user_items) == 0:
        return 0.0

    matched = user_items.intersection(career_items)

    return (len(matched) / len(user_items)) * 100


def calculate_requirement_match(user_level, required_level):
    """
    Compare student's skill level with career requirement.
    """

    levels = {
        "Low": 1,
        "Medium": 2,
        "High": 3
    }

    user_score = levels.get(user_level, 1)
    required_score = levels.get(required_level, 1)

    if user_score >= required_score:
        return 100.0

    if user_score == required_score - 1:
        return 60.0

    return 30.0


# ================================================================
# 4. MAIN RECOMMENDATION FUNCTION
# ================================================================

def recommend_careers(
    education,
    branch,
    interests,
    skills,
    programming,
    mathematics,
    communication,
    problem_solving,
    creativity,
    leadership,
    preferred_industry,
    experience
):
    """
    Generate career recommendations using hybrid scoring.
    """

    # ------------------------------------------------------------
    # Convert user inputs
    # ------------------------------------------------------------

    education = clean_text(education)
    branch = clean_text(branch)
    preferred_industry = clean_text(preferred_industry)
    experience = clean_text(experience)

    interests = {
        clean_text(x)
        for x in interests
        if clean_text(x)
    }

    skills = {
        clean_text(x)
        for x in skills
        if clean_text(x)
    }

    # ------------------------------------------------------------
    # Education + Branch combination
    # ------------------------------------------------------------

    user_education = f"{education} {branch}".strip()

    # ------------------------------------------------------------
    # Store recommendation results
    # ------------------------------------------------------------

    results = []

    # ============================================================
    # LOOP THROUGH EVERY CAREER
    # ============================================================

    for _, career in careers.iterrows():

        career_skills = convert_to_set(
            career["skills"]
        )

        career_interests = convert_to_set(
            career["interests"]
        )

        career_education = convert_to_set(
            career["education"]
        )

        career_industries = convert_to_set(
            career["industries"]
        )

        # --------------------------------------------------------
        # 1. SKILL MATCH
        # Weight = 35%
        # --------------------------------------------------------

        matching_skills = skills.intersection(
            career_skills
        )

        missing_skills = career_skills - skills

        if len(career_skills) > 0:
            skill_score = (
                len(matching_skills)
                / len(career_skills)
            ) * 100
        else:
            skill_score = 0

        # --------------------------------------------------------
        # 2. INTEREST MATCH
        # Weight = 20%
        # --------------------------------------------------------

        matching_interests = interests.intersection(
            career_interests
        )

        if len(career_interests) > 0:

            interest_score = (
                len(matching_interests)
                / len(career_interests)
            ) * 100

        else:
            interest_score = 0

        # --------------------------------------------------------
        # 3. EDUCATION / BRANCH MATCH
        # Weight = 15%
        # --------------------------------------------------------

        education_score = 0

        for item in career_education:

            if item in user_education:
                education_score = 100
                break

            # Handle common branch keywords
            if (
                "cse" in item
                and (
                    "computer" in branch
                    or "cse" in branch
                )
            ):
                education_score = 100
                break

            if (
                "aiml" in item
                and (
                    "artificial" in branch
                    or "machine learning" in branch
                    or "aiml" in branch
                )
            ):
                education_score = 100
                break

            if (
                "it" in item
                and "information technology" in branch
            ):
                education_score = 100
                break

        # Partial educational compatibility
        if education_score == 0:

            if education in [
                "b.tech",
                "m.tech",
                "mca",
                "b.sc"
            ]:
                education_score = 60

        # --------------------------------------------------------
        # 4. APTITUDE MATCH
        # Weight = 15%
        # --------------------------------------------------------

        programming_score = calculate_requirement_match(
            programming,
            career["programming_level"]
        )

        mathematics_score = calculate_requirement_match(
            mathematics,
            career["mathematics_level"]
        )

        communication_score = calculate_requirement_match(
            communication,
            career["communication_level"]
        )

        problem_solving_score = calculate_requirement_match(
            problem_solving,
            career["problem_solving_level"]
        )

        creativity_score = calculate_requirement_match(
            creativity,
            career["creativity_level"]
        )

        leadership_score = calculate_requirement_match(
            leadership,
            career["leadership_level"]
        )

        aptitude_score = np.mean([
            programming_score,
            mathematics_score,
            communication_score,
            problem_solving_score,
            creativity_score,
            leadership_score
        ])

        # --------------------------------------------------------
        # 5. INDUSTRY MATCH
        # Weight = 10%
        # --------------------------------------------------------

        if preferred_industry in career_industries:
            industry_score = 100
        else:
            industry_score = 30

        # --------------------------------------------------------
        # 6. EXPERIENCE MATCH
        # Weight = 5%
        # --------------------------------------------------------

        if experience == "student":
            experience_score = 100

        elif experience == "entry level":
            experience_score = 100

        elif experience == "mid level":

            if "mid" in career["career"].lower():
                experience_score = 100
            else:
                experience_score = 70

        elif experience == "senior level":

            if "senior" in career["career"].lower():
                experience_score = 100
            else:
                experience_score = 50

        else:
            experience_score = 70

        # ========================================================
        # 7. FINAL HYBRID SCORE
        # ========================================================

        final_score = (

            skill_score * 0.35

            + interest_score * 0.20

            + education_score * 0.15

            + aptitude_score * 0.15

            + industry_score * 0.10

            + experience_score * 0.05

        )

        # --------------------------------------------------------
        # 8. MATCHING JOBS FROM REAL DATASET
        # --------------------------------------------------------

        career_name = clean_text(
            career["career"]
        )

        # Keyword search based on career name
        career_keywords = career_name.split()

        matching_jobs = jobs[
            jobs["job_title"]
            .fillna("")
            .apply(
                lambda x:
                any(
                    keyword in clean_text(x)
                    for keyword in career_keywords
                    if len(keyword) > 3
                )
            )
        ]

        # --------------------------------------------------------
        # 9. SALARY INFORMATION
        # --------------------------------------------------------

        if len(matching_jobs) > 0:

            average_salary = (
                matching_jobs["salary"]
                .mean()
            )

            minimum_salary = (
                matching_jobs["salary"]
                .min()
            )

            maximum_salary = (
                matching_jobs["salary"]
                .max()
            )

        else:

            average_salary = jobs["salary"].mean()
            minimum_salary = jobs["salary"].min()
            maximum_salary = jobs["salary"].max()

        # --------------------------------------------------------
        # 10. SAVE RESULT
        # --------------------------------------------------------

        results.append({

            "career": career["career"],

            "category": career["category"],

            "match_score": round(
                final_score,
                2
            ),

            "skill_score": round(
                skill_score,
                2
            ),

            "interest_score": round(
                interest_score,
                2
            ),

            "education_score": round(
                education_score,
                2
            ),

            "aptitude_score": round(
                aptitude_score,
                2
            ),

            "industry_score": round(
                industry_score,
                2
            ),

            "matching_skills":
                ", ".join(
                    sorted(matching_skills)
                ),

            "missing_skills":
                ", ".join(
                    sorted(missing_skills)
                ),

            "average_salary":
                round(
                    average_salary,
                    2
                ),

            "minimum_salary":
                round(
                    minimum_salary,
                    2
                ),

            "maximum_salary":
                round(
                    maximum_salary,
                    2
                )
        })

    # ============================================================
    # 11. CREATE RESULT DATAFRAME
    # ============================================================

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by="match_score",
        ascending=False
    ).reset_index(drop=True)

    return result_df


# ================================================================
# 5. TEST THE RECOMMENDATION ENGINE
# ================================================================

test_result = recommend_careers(

    education="B.Tech",

    branch="AIML",

    interests=[
        "Artificial Intelligence",
        "Machine Learning",
        "Data Science"
    ],

    skills=[
        "Python",
        "SQL",
        "Machine Learning",
        "Research"
    ],

    programming="High",

    mathematics="High",

    communication="Medium",

    problem_solving="High",

    creativity="Medium",

    leadership="Medium",

    preferred_industry="Software",

    experience="Student"
)

# ================================================================
# 6. DISPLAY TOP 5
# ================================================================

print("\n" + "=" * 70)
print("TOP 5 CAREER RECOMMENDATIONS")
print("=" * 70)

display(
    test_result[
        [
            "career",
            "category",
            "match_score",
            "skill_score",
            "interest_score",
            "education_score",
            "aptitude_score",
            "industry_score"
        ]
    ].head(5)
)

# ================================================================
# 7. DISPLAY SKILL GAP
# ================================================================

print("\n" + "=" * 70)
print("TOP CAREER SKILL ANALYSIS")
print("=" * 70)

for i in range(min(5, len(test_result))):

    row = test_result.iloc[i]

    print("\n" + "-" * 60)

    print(
        f"Rank {i + 1}: {row['career']}"
    )

    print(
        f"Match Score: {row['match_score']}%"
    )

    print(
        f"Matching Skills: "
        f"{row['matching_skills']}"
    )

    print(
        f"Skills to Improve: "
        f"{row['missing_skills']}"
    )

    print(
        f"Average Salary: "
        f"{row['average_salary']}"
    )

print("\n" + "=" * 70)
print("✅ RECOMMENDATION ENGINE TEST COMPLETED!")
print("=" * 70)

HYBRID AI CAREER RECOMMENDATION ENGINE

Career Knowledge Base : (11, 13)
Job Dataset           : (50000, 13)

TOP 5 CAREER RECOMMENDATIONS


,career,category,match_score,skill_score,interest_score,education_score,aptitude_score,industry_score
0,AI / Machine Learning Engineer,Artificial Intelligence,78.33,66.67,50.0,100,100.00,100
1,Data Scientist,Data Science,78.33,66.67,50.0,100,100.00,100
2,Research Scientist,Research,77.00,80.00,25.0,100,93.33,100
3,Cybersecurity Analyst,Cybersecurity,66.00,60.00,0.0,100,100.00,100
4,Cloud / DevOps Engineer,Cloud Computing,59.00,40.00,0.0,100,100.00,100



TOP CAREER SKILL ANALYSIS

------------------------------------------------------------
Rank 1: AI / Machine Learning Engineer
Match Score: 78.33%
Matching Skills: machine learning, python, research, sql
Skills to Improve: aws, mathematics
Average Salary: 94971.88

------------------------------------------------------------
Rank 2: Data Scientist
Match Score: 78.33%
Matching Skills: machine learning, python, research, sql
Skills to Improve: excel, mathematics
Average Salary: 94840.61

------------------------------------------------------------
Rank 3: Research Scientist
Match Score: 77.0%
Matching Skills: machine learning, python, research, sql
Skills to Improve: mathematics
Average Salary: 94955.09

------------------------------------------------------------
Rank 4: Cybersecurity Analyst
Match Score: 66.0%
Matching Skills: python, research, sql
Skills to Improve: linux, problem solving
Average Salary: 92725.54

------------------------------------------------------------
Rank 5: C